# Basic Feast Example

#### To get started, run this in the command line:
\
feast init my_project   (only if my_project does not exist yet)
\
cd my_project/feature_repo
\
ls

**The resulting demo repo has the following components:**
\
data/ contains raw demo parquet data
\
example_repo.py contains demo feature definitions
\
test_workflpw.py showcases how to run all key Feast commands including defining, retrieving, and pushing features. You can run this with python test-workflow.py
\
feature_store.yaml contains a demo setup configuring where data sources are
\ valid values for provider in this file are:
* local: SQL registry or local file registry
* gcp: SQL registry or GCS file registry
* aws: use a SQL registry or S3 file registry


#### Imports

In [17]:
import pandas as pd
from datetime import datetime
import pandas as pd
from feast import FeatureStore
from pprint import pprint
import logging
logging.getLogger("feast").setLevel(logging.ERROR)

#### Inspect the Data

In [7]:
pd.read_parquet("my_project/feature_repo/data/driver_stats.parquet")

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2025-01-21 16:29:28.406872+00:00,1001,1.000000,1.000000,1000,2025-01-21 16:29:28.406874
1,2025-01-06 16:00:00+00:00,1005,0.504432,0.029913,627,2025-01-21 16:28:00.791000
2,2025-01-06 17:00:00+00:00,1005,0.822875,0.627012,891,2025-01-21 16:28:00.791000
3,2025-01-06 18:00:00+00:00,1005,0.825212,0.880664,253,2025-01-21 16:28:00.791000
4,2025-01-06 19:00:00+00:00,1005,0.288010,0.865955,708,2025-01-21 16:28:00.791000
...,...,...,...,...,...,...
1803,2025-01-21 14:00:00+00:00,1001,0.472953,0.993606,726,2025-01-21 16:28:00.791000
1804,2025-01-21 15:00:00+00:00,1001,0.762379,0.233713,573,2025-01-21 16:28:00.791000
1805,2021-04-12 07:00:00+00:00,1001,0.847427,0.734705,296,2025-01-21 16:28:00.791000
1806,2025-01-14 04:00:00+00:00,1003,0.239273,0.157236,467,2025-01-21 16:28:00.791000


#### Run Sample Workflow, Register Feature Definitions, Deploy Your Feature Store
In the command line: 
\
python3 test_workflow.py 
\
feast apply

#### Generate Training Data 

In [18]:
# Note: see https://docs.feast.dev/getting-started/concepts/feature-retrieval for 
# more details on how to retrieve for all entities in the offline store instead

# Create a DataFrame with entity and related data
entity_df = pd.DataFrame.from_dict(
    {
        # 'driver_id' is the primary key (entity) that will be used to fetch features.
        # These IDs correspond to the drivers for whom we want to retrieve historical features.
        "driver_id": [1001, 1002, 1003],
        
        # 'event_timestamp' is a special reserved column in Feast.
        # It indicates the timestamp of the event associated with the entity.
        # Feast uses this timestamp to fetch the features that were valid at the given time.
        "event_timestamp": [
            datetime(2021, 4, 12, 10, 59, 42),
            datetime(2021, 4, 12, 8, 12, 10),
            datetime(2021, 4, 12, 16, 40, 26),
        ],
        
        # 'label_driver_reported_satisfaction' is an optional column containing labels.
        # Labels are not processed by Feast but can be useful for supervised learning (e.g., target values for training).
        "label_driver_reported_satisfaction": [1, 5, 3],
        
        # These additional columns ('val_to_add' and 'val_to_add_2') are inputs for on-demand transformations.
        # On-demand transformations are calculated dynamically when retrieving features.
        "val_to_add": [1, 2, 3],
        "val_to_add_2": [10, 20, 30],
    }
)

# Initialize the FeatureStore.
# The 'repo_path' points to the directory containing the feature repository configuration.
# This repository defines how features are stored and retrieved.
store = FeatureStore(repo_path="my_project/feature_repo")

# Retrieve historical features for the entities defined in 'entity_df'.
# Historical features are fetched for the given entity keys and event timestamps.
# The features parameter specifies which features to retrieve.
training_df = store.get_historical_features(
    entity_df=entity_df,  # The input data specifying entities and timestamps
    features=[
        "driver_hourly_stats:conv_rate",          # Conversion rate feature
        "driver_hourly_stats:acc_rate",           # Acceptance rate feature
        "driver_hourly_stats:avg_daily_trips",    # Average daily trips feature
        "transformed_conv_rate:conv_rate_plus_val1",  # Transformed feature (on-demand transformation)
        "transformed_conv_rate:conv_rate_plus_val2",  # Another transformed feature
    ],
).to_df()  

print("----- Feature schema -----\n")
print(training_df.info())

print()
print("----- Example features -----\n")
print(training_df.head())

_list_feature_views will make breaking changes. Please use _list_batch_feature_views instead. _list_feature_views will behave like _list_all_feature_views in the future.


----- Feature schema -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   driver_id                           3 non-null      int64              
 1   event_timestamp                     3 non-null      datetime64[ns, UTC]
 2   label_driver_reported_satisfaction  3 non-null      int64              
 3   val_to_add                          3 non-null      int64              
 4   val_to_add_2                        3 non-null      int64              
 5   conv_rate                           3 non-null      float32            
 6   acc_rate                            3 non-null      float32            
 7   avg_daily_trips                     3 non-null      int32              
 8   conv_rate_plus_val1                 3 non-null      float64            
 9   conv_rate_plus_val2

#### Run Offline Inference

In [19]:
# Update the event_timestamp column in the entity DataFrame to the current time.
# This assigns the same current timestamp (in UTC) to all rows in the DataFrame.
# The pd.to_datetime("now", utc=True) function ensures the timestamp is in UTC format,
# which is a common requirement for Feast.
entity_df["event_timestamp"] = pd.to_datetime("now", utc=True)

# Retrieve historical features for the entities defined in the entity_df DataFrame.
# Since we updated the event_timestamp column to the current time, this will fetch the most recent features.
training_df = store.get_historical_features(
    entity_df=entity_df,  # Input DataFrame containing entities and timestamps.
    features=[
        "driver_hourly_stats:conv_rate",          # Conversion rate: feature from the "driver_hourly_stats" view.
        "driver_hourly_stats:acc_rate",           # Acceptance rate: another feature from the same view.
        "driver_hourly_stats:avg_daily_trips",    # Average daily trips: feature from the "driver_hourly_stats" view.
        "transformed_conv_rate:conv_rate_plus_val1",  # On-demand transformed feature using "val_to_add".
        "transformed_conv_rate:conv_rate_plus_val2",  # Another transformed feature using "val_to_add_2".
    ],
).to_df()  

print("\n----- Example features -----\n")
print(training_df.head())


_list_feature_views will make breaking changes. Please use _list_batch_feature_views instead. _list_feature_views will behave like _list_all_feature_views in the future.



----- Example features -----

   driver_id                  event_timestamp  \
0       1003 2025-01-21 16:46:13.642468+00:00   
1       1002 2025-01-21 16:46:13.642468+00:00   
2       1001 2025-01-21 16:46:13.642468+00:00   

   label_driver_reported_satisfaction  val_to_add  val_to_add_2  conv_rate  \
0                                   3           3            30    0.23611   
1                                   5           2            20    0.42814   
2                                   1           1            10    1.00000   

   acc_rate  avg_daily_trips  conv_rate_plus_val1  conv_rate_plus_val2  
0  0.206644              499              3.23611             30.23611  
1  0.812110              502              2.42814             20.42814  
2  1.000000             1000              2.00000             11.00000  


#### Ingest Batch Features Into Your Online Store

In the command line:

CURRENT_TIME=$(date -u +"%Y-%m-%dT%H:%M:%S")

feast materialize-incremental $CURRENT_TIME

This gets the current time in UTC format and then incrementally materializes data into the Feast feature store. Materialization is the process of ingesting feature data from offline stores into online stores up to the specified timestamp. This ensures that your online store has the most recent feature data

#### Fetch Feature Vectors for Inference

In [20]:
# 'store.get_online_features' is a function that allows us to retrieve real-time features from Feast's 
# online store, which is useful for serving features during model inference or making real-time predictions.

# Specify the features you want to retrieve:
feature_vector = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    
    # Define the entities for which you want to fetch these features.
    # Entity rows are provided as a list of dictionaries, where each dictionary represents one entity 
    # and includes the join key(s) with the corresponding entity value(s).
    entity_rows=[
        {"driver_id": 1004},
        {"driver_id": 1005},
    ],
    
    # Optional parameters could be included here if needed, such as filtering specific time ranges, etc.
).to_dict()  
pprint(feature_vector)

_list_feature_views will make breaking changes. Please use _list_batch_feature_views instead. _list_feature_views will behave like _list_all_feature_views in the future.
Cannot use sqlite_vec for vector search


{'acc_rate': [0.4303702116012573, 0.8106047511100769],
 'avg_daily_trips': [745, 454],
 'conv_rate': [0.9512690901756287, 0.8499264121055603],
 'driver_id': [1004, 1005]}


#### Use Feature Service to Fetch Online Features

In [21]:
# A FeatureService represents a logical grouping of features, typically associated with a specific business problem.
feature_service = store.get_feature_service("driver_activity_v1")

feature_vector = store.get_online_features(
    features=feature_service,
    entity_rows=[
        # Driver with ID 1001, along with the additional values for transformations or custom calculations.
        {
            "driver_id": 1001,         # Join key: 'driver_id' -> 1001
            "val_to_add": 1000,        # Custom value used for transformations (not processed by Feast directly)
            "val_to_add_2": 2000,      # Another custom value for transformations
        },
        
        # Driver with ID 1002, also with custom values.
        {
            "driver_id": 1002,         
            "val_to_add": 1001,        
            "val_to_add_2": 2002,      
        },
    ],
).to_dict()  
pprint(feature_vector)

_list_feature_views will make breaking changes. Please use _list_batch_feature_views instead. _list_feature_views will behave like _list_all_feature_views in the future.


{'conv_rate': [1.0, 0.4281395375728607],
 'conv_rate_plus_val1': [1001.0, 1001.4281395375729],
 'conv_rate_plus_val2': [2001.0, 2002.4281395375729],
 'driver_id': [1001, 1002]}
